# Session 11 — Knowledge modeling, querying & inference

Two small, self-contained PyTorch Geometric demos that you can run on a laptop:

**Part A — A heterogeneous (multi-relational) GCN.**
We build a tiny academic graph with three node types (papers, authors, venues)
and three relation types (`cites`, `writes`, `published_in`), then train an
**R-GCN** (Schlichtkrull et al. 2018 — the "Modeling Relational Data with GCNs"
reading from the heterogeneous-graphs lecture) to classify papers by topic from
only a handful of labels. We compare it against a *relation-blind* baseline to
see what the typed edges buy us.

**Part B — Knowledge-graph embeddings, inference and queries.**
We hand-build a small, fully human-readable knowledge graph and train **TransE**.
Because TransE learns the geometry `head + relation ≈ tail`, we can (1) *infer*
missing facts by link prediction, (2) answer 1-hop queries by ranking, and
(3) answer **2-hop queries directly in vector space** by adding relation vectors —
the intuition that the Query2box lecture builds on.

> The two parts use **independent datasets** and can be read in either order.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

---
## Part A — A heterogeneous / multi-relational GCN

### A.1  Build a synthetic academic graph

We lay every node (papers, authors, venues) into a single index space and
describe the graph with `edge_index` (`[2, E]`) and a parallel `edge_type`
(`[E]`) — exactly the format `RGCNConv` consumes. Topic structure is *planted*:
papers mostly cite within their topic, authors write within their topic, and
each paper is published in its topic's venue. None of this is given to the model
as a feature — it must recover it by message passing.

In [ ]:
rng = np.random.default_rng(7)

N_TOPICS, PAPERS_PER, AUTHORS_PER, VENUES_PER = 3, 20, 6, 1
n_paper  = N_TOPICS * PAPERS_PER
n_author = N_TOPICS * AUTHORS_PER
n_venue  = N_TOPICS * VENUES_PER
n_total  = n_paper + n_author + n_venue

paper_topic  = np.repeat(np.arange(N_TOPICS), PAPERS_PER)
author_topic = np.repeat(np.arange(N_TOPICS), AUTHORS_PER)
venue_topic  = np.repeat(np.arange(N_TOPICS), VENUES_PER)

# index layout: [ papers | authors | venues ]
pidx = lambda i: i
aidx = lambda i: n_paper + i
vidx = lambda i: n_paper + n_author + i

REL = {"cites": 0, "writes": 1, "published_in": 2}
src, dst, etype = [], [], []

def add(s, d, r):
    src.append(s); dst.append(d); etype.append(REL[r])

CITE_HOMOPHILY = 0.78
for p in range(n_paper):                       # paper --cites--> paper (homophilous)
    for _ in range(3):
        pool = np.where(paper_topic == paper_topic[p])[0] if rng.random() < CITE_HOMOPHILY \
               else np.arange(n_paper)
        q = int(rng.choice(pool))
        if q != p:
            add(pidx(p), pidx(q), "cites")
for a in range(n_author):                       # author --writes--> paper (same topic)
    for q in rng.choice(np.where(paper_topic == author_topic[a])[0], size=4, replace=False):
        add(aidx(a), pidx(int(q)), "writes")
for p in range(n_paper):                         # paper --published_in--> venue (same topic)
    v = int(rng.choice(np.where(venue_topic == paper_topic[p])[0]))
    add(pidx(p), vidx(v), "published_in")

# R-GCN treats relations as directed; add reverse edges as their own relation types
n_base_rel = len(REL)
S = np.array(src); D = np.array(dst); E = np.array(etype)
edge_index = torch.tensor(np.stack([np.concatenate([S, D]),
                                    np.concatenate([D, S])]), dtype=torch.long)
edge_type  = torch.tensor(np.concatenate([E, E + n_base_rel]), dtype=torch.long)
num_relations = 2 * n_base_rel

# labels exist only for paper nodes; -1 elsewhere ("not a paper / no label")
labels = torch.full((n_total,), -1, dtype=torch.long)
labels[:n_paper] = torch.tensor(paper_topic, dtype=torch.long)

# semi-supervised: reveal only 3 labels per topic
train_mask = torch.zeros(n_total, dtype=torch.bool)
for t in range(N_TOPICS):
    seeds = np.where(paper_topic == t)[0][:3]
    train_mask[seeds] = True
paper_mask = torch.zeros(n_total, dtype=torch.bool); paper_mask[:n_paper] = True
test_mask = paper_mask & ~train_mask

print(f"nodes: {n_total}  (papers {n_paper}, authors {n_author}, venues {n_venue})")
print(f"edges: {edge_index.size(1)}  | relation types (with reverses): {num_relations}")
print(f"labelled papers: {int(train_mask.sum())} of {n_paper}")

### A.2  The R-GCN

An R-GCN gives **each relation its own weight matrix**. A node's update sums
relation-specific messages from its neighbours:

$$h_i^{(l+1)} = \sigma\Big( W_0^{(l)} h_i^{(l)} + \sum_{r}\sum_{j\in\mathcal N_i^r}
  \tfrac{1}{|\mathcal N_i^r|}\, W_r^{(l)} h_j^{(l)} \Big)$$

A type-blind GCN collapses every $W_r$ into a single $W$ — our baseline below
is exactly that (one relation type), so the only difference is whether the model
is *allowed* to treat `cites`, `writes` and `published_in` differently.

In [ ]:
from torch_geometric.nn import RGCNConv

class RGCN(nn.Module):
    def __init__(self, num_nodes, hidden, num_classes, num_relations):
        super().__init__()
        self.emb   = nn.Embedding(num_nodes, hidden)     # learnable input features
        self.conv1 = RGCNConv(hidden, hidden, num_relations)
        self.conv2 = RGCNConv(hidden, num_classes, num_relations)

    def forward(self, edge_index, edge_type, return_hidden=False):
        x = self.emb.weight
        h = F.relu(self.conv1(x, edge_index, edge_type))
        out = self.conv2(h, edge_index, edge_type)
        return (out, h) if return_hidden else out

def train_model(num_relations, et, epochs=120, hidden=16, seed=0):
    torch.manual_seed(seed)
    model = RGCN(n_total, hidden, N_TOPICS, num_relations).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    ei, et = edge_index.to(DEVICE), et.to(DEVICE)
    y = labels.to(DEVICE); tr = train_mask.to(DEVICE)
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        out = model(ei, et)
        loss = F.cross_entropy(out[tr], y[tr])
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        out, h = model(ei, et, return_hidden=True)
        pred = out.argmax(1)
        acc = (pred[test_mask] == labels.to(DEVICE)[test_mask]).float().mean().item()
    return model, acc, h.cpu()

# Full R-GCN (sees all relation types) vs. a relation-blind baseline (1 type)
model_rgcn, acc_rgcn, hidden_rgcn = train_model(num_relations, edge_type)
blind_type = torch.zeros_like(edge_type)
model_blind, acc_blind, _ = train_model(1, blind_type)

print(f"R-GCN (relation-aware) test accuracy : {acc_rgcn:.2%}")
print(f"relation-blind baseline test accuracy: {acc_blind:.2%}")

### A.3  Watch the representation separate

We project the first hidden layer of the trained R-GCN to 2-D with PCA. Even
though only 3 papers per topic were labelled, message passing pulls each topic
into its own cluster.

In [ ]:
def pca2(x):
    x = x - x.mean(0, keepdim=True)
    u, s, v = torch.linalg.svd(x, full_matrices=False)
    return (x @ v[:2].T).numpy()

emb2 = pca2(hidden_rgcn[:n_paper])
fig, ax = plt.subplots(figsize=(6, 5))
for t in range(N_TOPICS):
    m = paper_topic == t
    ax.scatter(emb2[m, 0], emb2[m, 1], label=f"topic {t}", s=60, alpha=0.8)
seeds = train_mask[:n_paper].numpy()
ax.scatter(emb2[seeds, 0], emb2[seeds, 1], facecolors="none",
           edgecolors="k", s=160, linewidths=1.8, label="labelled")
ax.set_title(f"R-GCN paper representations (PCA)\ntest acc {acc_rgcn:.0%}  •  baseline {acc_blind:.0%}")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(loc="best", fontsize=8)
fig.tight_layout(); fig.savefig("fig_hetero_gcn.png", dpi=130)
plt.show()

**Takeaway.** Giving each relation its own transformation lets the model use
*who wrote* and *where it was published* differently from *what it cites*. On
real heterogeneous graphs that flexibility is what separates R-GCN-style models
from a single-matrix GCN.

---
## Part B — Knowledge-graph embeddings, inference & queries

### B.1  A readable knowledge graph

In [ ]:
triples_str = [
    ("Alice", "lives_in", "Trento"),  ("Bob", "lives_in", "Milan"),
    ("Carol", "lives_in", "Paris"),   ("Dan", "lives_in", "Lyon"),
    ("Trento", "located_in", "Italy"),("Milan", "located_in", "Italy"),
    ("Paris", "located_in", "France"),("Lyon", "located_in", "France"),
    ("Alice", "works_at", "Acme"),    ("Bob", "works_at", "Acme"),
    ("Carol", "works_at", "Globex"),  ("Dan", "works_at", "Globex"),
    ("Acme", "hq_in", "Milan"),       ("Globex", "hq_in", "Paris"),
    ("Italy", "capital", "Rome"),     ("France", "capital", "Paris"),
]
entities  = sorted({e for h, _, t in triples_str for e in (h, t)})
relations = sorted({r for _, r, _ in triples_str})
e2i = {e: i for i, e in enumerate(entities)}
r2i = {r: i for i, r in enumerate(relations)}
i2e = {i: e for e, i in e2i.items()}

head_index = torch.tensor([e2i[h] for h, _, _ in triples_str])
rel_type   = torch.tensor([r2i[r] for _, r, _ in triples_str])
tail_index = torch.tensor([e2i[t] for _, _, t in triples_str])
print(f"{len(entities)} entities, {len(relations)} relations, {len(triples_str)} triples")

### B.2  Train TransE — and yes, it is *learned*

We use PyTorch Geometric's `TransE` model. Nothing here is hand-set: the model
holds two trainable `Embedding` tables — one row per **entity**
(`kge.node_emb`) and one per **relation** (`kge.rel_emb`) — initialised
randomly and then learned by gradient descent.

TransE scores a triple as $-\lVert \mathbf h + \mathbf r - \mathbf t\rVert$, so
a fact is plausible when *head + relation* lands near *tail*. Training pushes
the embeddings to satisfy that geometry on the true triples while pushing
corrupted (false) triples apart, via a **margin ranking loss**:

$$\mathcal L = \sum_{(h,r,t)} \big[\,\gamma + d(h,r,t) - d(h',r,t')\,\big]_+$$

PyG does the heavy lifting for us:

- `kge.loader(...)` iterates over the true triples in minibatches and, for each
  positive triple, **samples a corrupted negative** (a random wrong head/tail);
- `kge.loss(h, r, t)` scores the positives and those negatives and returns the
  margin loss above;
- a standard `torch.optim.Adam` step updates both embedding tables.

After the loop, the learned vectors live in `kge.node_emb.weight` and
`kge.rel_emb.weight` — that is what every query below reads from. We print the
loss every 100 epochs; it should fall steadily as the geometry is learned.

In [ ]:
from torch_geometric.nn import TransE

kge = TransE(num_nodes=len(entities), num_relations=len(relations),
             hidden_channels=16).to(DEVICE)
loader = kge.loader(head_index.to(DEVICE), rel_type.to(DEVICE),
                    tail_index.to(DEVICE), batch_size=8, shuffle=True)
opt = torch.optim.Adam(kge.parameters(), lr=0.01)

for epoch in range(1, 401):
    kge.train(); total = 0.0
    for h, r, t in loader:
        opt.zero_grad()
        loss = kge.loss(h, r, t)
        loss.backward(); opt.step()
        total += float(loss)
    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}  loss {total/len(loader):.4f}")

### B.3  Inference by link prediction

Rank every entity as the tail of a query `(head, relation, ?)` using the model's
own scoring. The true tail should float to the top — and *plausible* tails the
graph never stated should still score better than nonsense.

In [ ]:
@torch.no_grad()
def rank_tails(head, relation, k=4):
    h = torch.full((len(entities),), e2i[head], device=DEVICE)
    r = torch.full((len(entities),), r2i[relation], device=DEVICE)
    cand = torch.arange(len(entities), device=DEVICE)
    scores = kge(h, r, cand)                 # higher = better
    order = scores.argsort(descending=True).cpu().tolist()
    out = [(i2e[i], float(scores[i])) for i in order if i2e[i] != head][:k]
    return out

for q in [("Alice", "works_at"), ("Trento", "located_in"), ("Acme", "hq_in")]:
    print(f"{q[0]} —{q[1]}→ ?")
    for name, s in rank_tails(*q):
        print(f"     {name:8s}  score {s:+.3f}")

Optional aggregate metrics over the known triples (filtered ranking):

In [ ]:
kge.eval()
rank, mrr, hits10 = kge.test(head_index.to(DEVICE), rel_type.to(DEVICE),
                             tail_index.to(DEVICE), batch_size=16, k=10)
print(f"mean rank {rank:.2f}  |  MRR {mrr:.3f}  |  Hits@10 {hits10:.3f}")

### B.4  A 2-hop query *in vector space*

The graph never stores which **country** a person lives in — only
`person —lives_in→ city` and `city —located_in→ country`. With TransE we can
*compose* the two relations by adding their vectors:

$$\mathbf{person} + \mathbf r_{\text{lives\_in}} + \mathbf r_{\text{located\_in}}
  \;\approx\; \mathbf{country}$$

This is the seed of the idea the **Query2box** lecture develops: answer
conjunctive multi-hop queries by doing geometry on embeddings instead of
traversing the graph.

In [ ]:
@torch.no_grad()
def compose_query(head, rel_chain, k=4):
    p = 1  # TransE p-norm
    node = F.normalize(kge.node_emb.weight, p=p, dim=-1)
    q = node[e2i[head]].clone()
    for r in rel_chain:
        q = q + kge.rel_emb.weight[r2i[r]]
    dist = (node - q).norm(p=p, dim=-1)       # smaller = better
    order = dist.argsort().cpu().tolist()
    return [(i2e[i], float(dist[i])) for i in order if i2e[i] != head][:k]

is_stored = lambda tr: tr in triples_str
for person in ["Alice", "Bob", "Carol", "Dan"]:
    top = compose_query(person, ["lives_in", "located_in"])
    best = top[0][0]
    print(f"{person} lives_in → located_in  ⇒  {best:7s}   "
          f"(stored as a triple? {is_stored((person, 'lives_in→located_in', best))})")
    print("       ranking:", ", ".join(f"{n}:{d:.2f}" for n, d in top))

### B.5  See the geometry

PCA of the entity embeddings with the learned `lives_in` translation drawn as
arrows: every person sits roughly one `lives_in` vector away from their city.

In [ ]:
@torch.no_grad()
def emb_np():
    return F.normalize(kge.node_emb.weight, p=1, dim=-1).cpu().numpy()

X = emb_np()
Xc = X - X.mean(0, keepdims=True)
_, _, V = np.linalg.svd(Xc, full_matrices=False)
P = Xc @ V[:2].T

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(P[:, 0], P[:, 1], s=40, color="#888")
for name, i in e2i.items():
    ax.annotate(name, (P[i, 0], P[i, 1]), fontsize=9,
                xytext=(4, 4), textcoords="offset points")
for h, r, t in triples_str:
    if r == "lives_in":
        ax.annotate("", xy=(P[e2i[t], 0], P[e2i[t], 1]),
                    xytext=(P[e2i[h], 0], P[e2i[h], 1]),
                    arrowprops=dict(arrowstyle="->", color="#c0392b", alpha=0.7))
ax.set_title("TransE entity embeddings (PCA)\nred arrows = learned 'lives_in' translation")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
fig.tight_layout(); fig.savefig("fig_kg_transe.png", dpi=130)
plt.show()

**Takeaways.**

- **Embeddings turn knowledge into geometry.** Plausible facts are short
  `h + r → t` hops; implausible ones are far. That is *inference*, not lookup.
- **Queries become vector arithmetic.** A 2-hop question is answered by adding
  relation vectors — the intuition behind box/region query embeddings
  (Query2box) and behind GNN recommenders that score user→item links.
- **Where this goes next.** Inductive and *foundation* KG models (e.g. ULTRA)
  keep this scoring idea but learn relation representations that transfer to
  knowledge graphs — and relations — they have never seen.